# kanbior-tts: first VITS run (Mac smoke test)

Trains a small multi-speaker **VITS** model ([coqui-tts](https://github.com/idiap/coqui-ai-TTS), MPL-2.0,
trained from scratch: no MMS / non-commercial weights) on a few JML speakers, on the CPU.

This is a **pipeline check**, like the ASR smoke run: after a few hundred steps the output is noise or
babble. It verifies the data, the character set with tones, training, checkpoints and synthesis
before a long GPU run on Kaggle (VITS needs ~100k+ steps to sound good).

Before running:
1. `python prepare_data.py` (from `kanbior-tts/`) to write `data/train.csv` and `data/val.csv`.
2. Install coqui-tts in the activated env: `uv pip install "coqui-tts[codec]" six`
   (`six` is a missing dependency of coqui-tts 0.28).

In [1]:
RUN = "tiny"   # or "tiny"

PRESETS = {
    # 200 steps on CPU: checks the pipeline end to end (~6-8 s/step: ~20-30 min on an M2)
    "smoke": dict(n_speakers=4, max_train=400, epochs=4, batch_size=8),
    # ~5 steps: checks that everything runs, on a small Mac (8 GB RAM): ~1 min
    "tiny": dict(n_speakers=2, max_train=24, epochs=1, batch_size=4),
}
cfg = PRESETS[RUN]

from pathlib import Path

DATA_DIR = Path("../kanbior-asr/sample-data")   # transcribed audio, shared with the ASR
SPLITS_DIR = Path("data")                        # written by prepare_data.py
OUTPUT_DIR = Path("runs") / f"vits-fon-{RUN}"
SAMPLE_RATE = 16_000                             # JML audio is 16 kHz
SEED = 42

In [2]:
import csv
import random
import sys
import unicodedata
from collections import Counter

import torch
from IPython.display import Audio, display
from TTS.config.shared_configs import BaseAudioConfig
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.configs.vits_config import VitsArgs, VitsConfig
from TTS.tts.models.vits import Vits
from trainer import Trainer, TrainerArgs

torch.manual_seed(SEED)
print("torch", torch.__version__)

torch 2.15.0


## 1. Data

Text goes in as **characters, with tones** (Fon spelling is close to pronunciation, so no
phonemizer). It is lowercased and NFD-normalized, so each tone mark is its own token shared
across vowels: `ɔ́` -> `ɔ` + `◌́`.

In [3]:
PUNCTUATIONS = "!'(),-.:;? "

def tts_text(text: str) -> str:
    return unicodedata.normalize("NFD", text.lower())

def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

train_rows, val_rows = read_csv(SPLITS_DIR / "train.csv"), read_csv(SPLITS_DIR / "val.csv")

# Smoke run: the speakers with the most clips in the train list.
speakers = [s for s, _ in Counter(r["speaker_id"] for r in train_rows).most_common(cfg["n_speakers"])]
train_rows = [r for r in train_rows if r["speaker_id"] in speakers]
val_rows = [r for r in val_rows if r["speaker_id"] in speakers]
random.Random(SEED).shuffle(train_rows)
train_rows = train_rows[: cfg["max_train"]]
if len(val_rows) < 4:   # few smoke speakers may have no val clips: borrow some train clips
    val_rows, train_rows = val_rows + train_rows[:8], train_rows[8:]

def to_sample(r):
    return {
        "text": tts_text(r["text"]),
        "audio_file": str((DATA_DIR / r["audio_path"]).resolve()),
        "speaker_name": r["speaker_id"],
        "language": None,
        "root_path": str(DATA_DIR.resolve()),
        "audio_unique_name": r["id"],
    }

train_samples, eval_samples = [to_sample(r) for r in train_rows], [to_sample(r) for r in val_rows]
hours = sum(float(r["duration"]) for r in train_rows) / 3600
print(f"Speakers: {speakers}")
print(f"train {len(train_samples)} clips ({hours:.2f} h), val {len(eval_samples)} clips")

characters = sorted({ch for s in train_samples + eval_samples for ch in s["text"]} - set(PUNCTUATIONS))
print(f"{len(characters)} characters:", " ".join(unicodedata.name(c, c) if unicodedata.category(c) == "Mn" else c
                                          for c in characters))

Speakers: ['JML_S082', 'JML_S041']
train 24 clips (0.05 h), val 15 clips
32 characters: a b c d e f g h i j k l m n o p q r s t u v w x y z ɔ ɖ ɛ COMBINING GRAVE ACCENT COMBINING ACUTE ACCENT COMBINING CARON


## 2. Model and config

In [4]:
audio_config = BaseAudioConfig(
    sample_rate=SAMPLE_RATE, fft_size=1024, win_length=1024, hop_length=256,
    num_mels=80, mel_fmin=0, mel_fmax=None,
)
test_rows = val_rows[:2]
config = VitsConfig(
    model_args=VitsArgs(use_speaker_embedding=True),
    audio=audio_config,
    run_name=f"vits-fon-{RUN}",
    output_path=str(OUTPUT_DIR),
    batch_size=cfg["batch_size"],
    eval_batch_size=cfg["batch_size"],
    num_loader_workers=0,          # notebooks on macOS: keep data loading in the main process
    num_eval_loader_workers=0,
    epochs=cfg["epochs"],
    print_step=10,
    print_eval=False,
    run_eval=True,
    test_delay_epochs=-1,
    mixed_precision=False,
    text_cleaner=None,             # text is already cleaned by prepare_data.py + tts_text()
    use_phonemes=False,
    add_blank=True,
    characters=CharactersConfig(
        characters_class="TTS.tts.models.vits.VitsCharacters",
        characters="".join(characters),
        punctuations=PUNCTUATIONS,
        pad="<PAD>",
        phonemes="",
        is_unique=True,
        is_sorted=True,
    ),
    test_sentences=[[tts_text(r["text"]), r["speaker_id"], None, None] for r in test_rows],
    # Smoke run: no intermediate checkpoints (~1 GB each with optimizer state); the final
    # weights are saved in section 4. For long runs, set e.g. save_step=5000, save_best_after=0.
    save_step=10**9,
    save_n_checkpoints=1,
    save_best_after=10**9,
)

config.speakers = sorted(speakers)
config.model_args.num_speakers = len(speakers)

model = Vits(config)   # builds its tokenizer, audio processor and speaker table from the config
tokenizer = model.tokenizer
print(f"VITS: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M parameters, "
      f"{model.num_speakers} speakers, {tokenizer.characters.num_chars} tokens")

# Tone marks must survive tokenization (no unknown-character drops).
example = train_samples[0]["text"]
assert tokenizer.decode(tokenizer.encode(example)) == example, "characters lost in tokenization"
print("Tokenizer round-trip OK:", unicodedata.normalize("NFC", example))

/Users/dinyad/Documents/IA/kanbior/.kanbior-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Could not identify language from config. Initializing with English for text processing.


VITS: 86.4 M parameters, 2 speakers, 45 tokens
Tokenizer round-trip OK: a nǎ nu atínkɛn élɔ ɔ́ nǔgbǒ nú a ba ɖɔ azɔn émítɔn ní gbɔ ɔ́ nɛ́.


## 3. Train

Runs on the CPU (the coqui trainer supports CUDA or CPU). Logs, `config.json` and `speakers.pth`
go to `runs/vits-fon-smoke/<run>/`. TensorBoard: `tensorboard --logdir runs`.

In [5]:
trainer = Trainer(
    TrainerArgs(),
    config,
    str(OUTPUT_DIR),
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
    parse_command_line_args=False,   # we're in a notebook, not a script
)

 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Num. of CPUs: 8
 | > Num. of Torch Threads: 4
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=runs/vits-fon-tiny/vits-fon-tiny-November-23-2026_12+19AM-a8d72b1

 > Model has 86394412 parameters


In [6]:
trainer.fit()


 > EPOCH: 0/0
 --> runs/vits-fon-tiny/vits-fon-tiny-November-23-2026_12+19AM-a8d72b1
100%|██████████| 24/24 [00:00<00:00, 2346.74it/s]

 > TRAINING (2026-11-23 00:19:16) 

   --> TIME: 2026-11-23 00:19:24 -- STEP: 0/5 -- GLOBAL_STEP: 0
     | > current_lr_0: 0.0002  (0.0002)
     | > current_lr_1: 0.0002  (0.0002)
     | > loss_disc: 5.952507972717285  (5.952507972717285)
     | > loss_disc_real_0: 1.0435925722122192  (1.0435925722122192)
     | > loss_disc_real_1: 0.9705124497413635  (0.9705124497413635)
     | > loss_disc_real_2: 0.9544700384140015  (0.9544700384140015)
     | > loss_disc_real_3: 1.0063399076461792  (1.0063399076461792)
     | > loss_disc_real_4: 0.9811766147613525  (0.9811766147613525)
     | > loss_disc_real_5: 0.9950883984565735  (0.9950883984565735)
     | > loss_0: 5.952507972717285  (5.952507972717285)
     | > grad_norm_0: tensor(6.7822)  (tensor(6.7822))
     | > loss_gen: 4.336906909942627  (4.336906909942627)
     | > loss_kl: 209.45460510253906  (209.4546

## 4. Listen

Saves the final weights, reloads them (checks that saving and loading work) and synthesizes val sentences.
After a smoke run, expect noise or babble with roughly the right length and rhythm.

In [7]:
from TTS.utils.synthesizer import Synthesizer

# Weights only (~330 MB). Full trainer checkpoints also hold optimizer state (~1 GB each).
run_dir = Path(trainer.output_path)
checkpoint = run_dir / "model_final.pth"
torch.save({"model": trainer.model.state_dict()}, checkpoint)
synth = Synthesizer(
    tts_checkpoint=str(checkpoint),
    tts_config_path=str(run_dir / "config.json"),
    tts_speakers_file=str(run_dir / "speakers.pth"),
)
print("Loaded", checkpoint)

for r in val_rows[:3]:
    print(f"\n{r['id']}  {r['speaker_id']}:  {r['text']}")
    print("  reference:")
    display(Audio(str(DATA_DIR / r["audio_path"])))
    wav = synth.tts(tts_text(r["text"]), speaker_name=r["speaker_id"], split_sentences=False)
    print("  synthesized:")
    display(Audio(wav, rate=SAMPLE_RATE))

Loaded runs/vits-fon-tiny/vits-fon-tiny-November-23-2026_12+19AM-a8d72b1/model_final.pth

tr_000585  JML_S082:  Armen nɔ byɔ́ sinsɛnxɔmɛ tɛgbɛ bó nɔ́ xo ɖɛ azǎn lɛ́ɛ bǐ gbe.
  reference:


  synthesized:



tr_001154  JML_S041:  Jeanne sín nɔ ɔ́, nǔsatɔ́ ɖaxó ɖokpó wɛ é nyí ɖo aximɛ. É nɔ́ sa atínsínsɛ́n kpódó nǔ ɖěvo lɛ́ɛ kpó.
  reference:


  synthesized:



tr_001595  JML_S041:  Quartier lɛ́vɔ́ wá gbɛ, hwenu e gan wǒ zǎnzǎn tɔn xo tlóló é.
  reference:


  synthesized:
